In [7]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import time
from collections import Counter
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True

Using device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [8]:
train_df = pd.read_csv("../data/processed/train_all_versions.csv")
val_df = pd.read_csv("../data/processed/val_all_versions.csv")
test_df = pd.read_csv("../data/processed/test_all_versions.csv")

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(155551, 8)
(19444, 8)
(19444, 8)


In [15]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\monis\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\monis\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

Convert Sentiment to Numeric

In [9]:
print(train_df["sentiment"].unique())


<StringArray>
['very_positive', 'positive', 'neutral', 'negative', 'very_negative']
Length: 5, dtype: str


In [10]:
label_mapping = {
    "very_negative": 0,
    "negative": 1,
    "neutral": 2,
    "positive": 3,
    "very_positive": 4
}

train_df["label"] = train_df["sentiment"].map(label_mapping)
val_df["label"] = val_df["sentiment"].map(label_mapping)
test_df["label"] = test_df["sentiment"].map(label_mapping)

# Drop missing
train_df = train_df.dropna(subset=["text_dl", "label"])
val_df = val_df.dropna(subset=["text_dl", "label"])
test_df = test_df.dropna(subset=["text_dl", "label"])

EXTRACT TEXT + LABELS

In [12]:
train_sentences = train_df["text_dl"].astype(str).values
val_sentences = val_df["text_dl"].astype(str).values
test_sentences = test_df["text_dl"].astype(str).values

train_labels = train_df["label"].values
val_labels = val_df["label"].values
test_labels = test_df["label"].values


In [13]:
print(train_df["sentiment"].value_counts())


sentiment
very_positive    86881
positive         31980
neutral          17148
very_negative    10613
negative          8847
Name: count, dtype: int64


Build Vocabulary (ONLY from train)

Remove Rare Words

In [17]:
words = Counter()

for sentence in train_sentences:
    tokens = sentence.split()
    words.update(tokens)

min_freq = 3

vocab = [word for word, freq in words.items() if freq >= min_freq]
vocab = ["_PAD", "_UNK"] + vocab
word2idx = {word: idx for idx, word in enumerate(vocab)}

print("Vocab size:", len(vocab))


Vocab size: 64671


Encode Train/Val/Test

In [18]:
def encode_sentences(sentences):
    encoded = []
    for sentence in sentences:
        tokens = sentence.split()
        encoded.append([word2idx.get(word, 1) for word in tokens])
    return encoded

train_encoded = encode_sentences(train_sentences)
val_encoded = encode_sentences(val_sentences)
test_encoded = encode_sentences(test_sentences)


Padding

In [19]:
seq_len = 200

def pad_sequences(encoded_sentences):
    features = np.zeros((len(encoded_sentences), seq_len), dtype=np.int64)
    for i, review in enumerate(encoded_sentences):
        features[i, -len(review):] = np.array(review)[:seq_len]
    return features

X_train = pad_sequences(train_encoded)
X_val = pad_sequences(val_encoded)
X_test = pad_sequences(test_encoded)


Convert to PyTorch Tensors

In [20]:
X_train = torch.from_numpy(X_train)
X_val = torch.from_numpy(X_val)
X_test = torch.from_numpy(X_test)

y_train = torch.from_numpy(train_labels).long()
y_val = torch.from_numpy(val_labels).long()
y_test = torch.from_numpy(test_labels).long()


C:\Users\monis\AppData\Local\Temp\ipykernel_55424\1065911882.py:5: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\torch\csrc\utils\tensor_numpy.cpp:212.)
  y_train = torch.from_numpy(train_labels).long()


Create DataLoaders

In [22]:
batch_size = 256  

train_data = TensorDataset(X_train, y_train)
val_data = TensorDataset(X_val, y_val)
test_data = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_data, batch_size=batch_size, pin_memory=True)
test_loader = DataLoader(test_data, batch_size=batch_size, pin_memory=True)


Handle Class Imbalance

In [23]:
class_counts = train_df["sentiment"].value_counts().sort_index()

total = sum(class_counts)
num_classes = len(class_counts)

weights = [total / (num_classes * count) for count in class_counts]
weights = torch.tensor(weights, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=weights)


Define Bidirectional LSTM

In [24]:
class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, n_layers, output_size):
        super(BiLSTM, self).__init__()
        
        self.n_layers = n_layers
        self.hidden_dim = hidden_dim
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            n_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )
        
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim * 2, output_size)
        
    def forward(self, x, hidden):
        embeds = self.embedding(x)
        lstm_out, hidden = self.lstm(embeds, hidden)
        out = self.dropout(lstm_out[:, -1, :])
        out = self.fc(out)
        return out, hidden
    
    def init_hidden(self, batch_size):
        return (
            torch.zeros(self.n_layers * 2, batch_size, self.hidden_dim).to(device),
            torch.zeros(self.n_layers * 2, batch_size, self.hidden_dim).to(device)
        )


Initialize Model

In [25]:
model = BiLSTM(
    vocab_size=len(vocab),
    embedding_dim=300,
    hidden_dim=256,
    n_layers=2,
    output_size=5
)

model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


Training Loop (With Validation)

In [26]:
epochs = 5
clip = 5
best_val_loss = float("inf")
patience = 2
early_stop_counter = 0

for epoch in range(epochs):
    
    start_time = time.time()
    print(f"\nEpoch {epoch+1}/{epochs}")
    
    model.train()
    train_loss = 0
    correct = 0
    total_samples = 0
    
    for inputs, labels in train_loader:
        
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        h = tuple([each.detach() for each in model.init_hidden(inputs.size(0))])
        
        optimizer.zero_grad()
        outputs, _ = model(inputs, h)
        loss = criterion(outputs, labels)
        
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        
        train_loss += loss.item()
        
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total_samples += labels.size(0)
    
    train_acc = 100 * correct / total_samples
    
    # Validation
    model.eval()
    val_loss = 0
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            h = model.init_hidden(inputs.size(0))
            outputs, _ = model(inputs, h)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {avg_val_loss:.4f}")
    print(f"Time: {(time.time()-start_time)/60:.2f} mins")
    
    if avg_val_loss < best_val_loss:
        torch.save(model.state_dict(), "best_bilstm_model.pt")
        best_val_loss = avg_val_loss
        early_stop_counter = 0
    else:
        early_stop_counter += 1
        if early_stop_counter >= patience:
            print("Early stopping triggered.")
            break



Epoch 1/5
Train Loss: 1.1474
Train Acc: 29.84%
Val Loss: 1.0149
Time: 1.52 mins

Epoch 2/5
Train Loss: 0.9373
Train Acc: 42.18%
Val Loss: 0.9471
Time: 1.53 mins

Epoch 3/5
Train Loss: 0.8223
Train Acc: 48.96%
Val Loss: 0.9072
Time: 1.52 mins

Epoch 4/5
Train Loss: 0.7131
Train Acc: 54.75%
Val Loss: 0.9756
Time: 1.50 mins

Epoch 5/5
Train Loss: 0.5956
Train Acc: 61.43%
Val Loss: 1.1104
Time: 1.50 mins
Early stopping triggered.


Full Evaluation

In [27]:
model.load_state_dict(torch.load("best_bilstm_model.pt"))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        h = model.init_hidden(inputs.size(0))
        outputs, _ = model(inputs, h)
        _, predicted = torch.max(outputs, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(classification_report(all_labels, all_preds, digits=4))
print(confusion_matrix(all_labels, all_preds))


C:\Users\monis\AppData\Local\Temp\ipykernel_55424\2093521637.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_bilstm_model.pt"))


              precision    recall  f1-score   support

           0     0.4479    0.7801    0.5691      1328
           1     0.3113    0.1148    0.1678      1106
           2     0.5378    0.1525    0.2376      2144
           3     0.2986    0.8446    0.4412      3995
           4     0.9149    0.4049    0.5614     10861

    accuracy                         0.4766     19434
   macro avg     0.5021    0.4594    0.3954     19434
weighted avg     0.6804    0.4766    0.4791     19434

[[1036   63   40  174   15]
 [ 538  127  103  325   13]
 [ 356  151  327 1288   22]
 [ 121   42   99 3374  359]
 [ 262   25   39 6137 4398]]
